# LangSmith Course: Tracing, Testing, & Evaluating Agentic Workflows
Welcome to the LangSmith course! In this lab, we will learn how to monitor, debug, and quantitatively evaluate our two complex agents:
1. **Email Agent** (Categorization & Structured checklist management)
2. **Code Writer Agent** (Self-healing local code compiler)

### What we will cover:
1. **Auto-Tracing & Visualizing Graphs**: Inspecting the nested run execution tree in the LangSmith UI.
2. **Metadata & Custom Run Tags**: Labeling and categorizing runs to isolate test sessions.
3. **LangSmith Prompt Hub**: Storing and versioning our system instructions in the cloud.
4. **Programmatic Datasets**: Constructing reference datasets of sample customer emails and coding prompts.
5. **Custom Evaluators & Benchmarking**: Running automated code execution and semantic audits against our agents, returning metrics directly to the LangSmith dashboard.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("LangChain Tracing:", os.environ.get("LANGCHAIN_TRACING_V2"))
print("LangSmith API Key loaded:", "LANGCHAIN_API_KEY" in os.environ)
print("Project Name:", os.environ.get("LANGCHAIN_PROJECT"))

## Part 1: Auto-Tracing & Run Meta-Tags
Because your `.env` contains `LANGCHAIN_TRACING_V2=true`, LangGraph automatically sends full tracing metadata to LangSmith on every invocation. 

We can add custom tag names and run metadata keys when we call `invoke` or `stream`. This allows us to organize, search, and filter runs in the LangSmith console.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class SimpleState(TypedDict):
    text: str

def sample_node(state: SimpleState):
    return {"text": state["text"].upper()}

builder = StateGraph(SimpleState)
builder.add_node("uppercaser", sample_node)
builder.add_edge(START, "uppercaser")
builder.add_edge("uppercaser", END)
graph = builder.compile()

# We run the graph, passing metadata and custom tags in the config dictionary!
config = {
    "metadata": {
        "agent_version": "1.0.0",
        "environment": "development",
        "user_id": "dev_user_12"
    },
    "tags": ["essential_labs", "quick_run"]
}

res = graph.invoke({"text": "hello langsmith!"}, config)
print("Result:", res)
print("Check your LangSmith Project Dashboard under 'langgraph-learning' to view the tagged run tree!")

## Part 2: Versioning Prompts using LangSmith Hub
Instead of hardcoding system instructions inside your notebook cells, you can host them on the **LangSmith Hub**. This allows you to update, test, and version prompts in the cloud without modifying any python scripts.

Let's see how to pull a prompt template from the LangSmith Hub. 
*(Note: You can pull public prompts without authentication, or private ones using your API key)*.

In [ ]:
from langsmith import Client
from langchain_openai import ChatOpenAI

try:
    # We pull a public general RAG prompt from the LangSmith Hub
    client = Client()
    prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)
    print("Successfully pulled prompt template from Hub:")
    print(prompt)
except Exception as e:
    print("Could not pull prompt (Check connectivity/Hub configurations):", e)

## Part 3: Programmatic Datasets for EmailAgent
To evaluate an agent, we need a **Dataset**. A Dataset in LangSmith is a collection of inputs and expected outputs (targets).

Let's create an evaluation dataset for the `EmailAgent` classification node. We'll populate it with typical customer emails and their expected classifications (Billing, Technical, Feedback, or General).

In [ ]:
from langsmith import Client

client = Client()
dataset_name = "Email Classification Benchmark"

# Sample inputs and targets for support emails
examples = [
    (
        {"subject": "Unsubscribe", "body": "Can you refund my subscription renewal card charge?"},
        {"category": "billing"}
    ),
    (
        {"subject": "Database connection timed out", "body": "I am getting a connection limit error on port 5432."},
        {"category": "technical"}
    ),
    (
        {"subject": "Feature request", "body": "It would be great if you added a CSV export button on the tables."},
        {"category": "feedback"}
    ),
    (
        {"subject": "Office hours", "body": "Where are you located and when is the support desk open?"},
        {"category": "general"}
    )
]

# Check if dataset already exists, otherwise create it
if client.has_dataset(dataset_name=dataset_name):
    print(f"Dataset '{dataset_name}' already exists. Retrieving it.")
    dataset = client.read_dataset(dataset_name=dataset_name)
else:
    print(f"Creating new dataset: {dataset_name}")
    dataset = client.create_dataset(dataset_name=dataset_name, description="Validation benchmarks for the email categorizer node.")
    
    for input_val, output_val in examples:
        client.create_example(
            inputs=input_val,
            outputs=output_val,
            dataset_id=dataset.id
        )
    print("Populated dataset with validation inputs.")

## Part 4: Creating a CodeWriter Dataset
Now let's build a testing dataset for the **CodeWriterAgent**. The inputs will be code description prompts, and the target will be standard code strings or expected method names to evaluate against.

In [ ]:
code_dataset_name = "Code Generator Benchmark"

code_examples = [
    (
        {"task_description": "Write a function named 'multiply(a, b)' that multiplies a and b."},
        {"method_name": "multiply"}
    ),
    (
        {"task_description": "Write a function named 'reverse_list(lst)' that returns the list reversed."},
        {"method_name": "reverse_list"}
    )
]

if client.has_dataset(dataset_name=code_dataset_name):
    print(f"Dataset '{code_dataset_name}' already exists.")
    code_dataset = client.read_dataset(dataset_name=code_dataset_name)
else:
    print(f"Creating new dataset: {code_dataset_name}")
    code_dataset = client.create_dataset(dataset_name=code_dataset_name, description="Benchmarks for evaluating code generation agent.")
    for input_val, output_val in code_examples:
        client.create_example(
            inputs=input_val,
            outputs=output_val,
            dataset_id=code_dataset.id
        )
    print("Populated Code Generator dataset.")

## Part 5: Running Evaluations
LangSmith enables us to write custom evaluation functions. These functions take the agent's run output, compare it against the expected dataset target, and output a float score (e.g. `0.0` or `1.0`).

### 1. Evaluating the Email Categorizer
We define a target function (our categorizer node) and an evaluator verifying if the classification matches the expected category label.

In [ ]:
from mock_llm import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langsmith.evaluation import evaluate
import json

# Target Runner: The function we are testing
def email_classifier_target(inputs: dict):
    llm = get_llm(model="gpt-4o-mini", temperature=0)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Classify email subject/body into category: 'billing', 'technical', 'feedback', 'general'. Return response ONLY as JSON: {{'category': '...'}}"),
        ("human", "Subject: {subject}\nBody: {body}")
    ])
    chain = prompt | llm
    res = chain.invoke({"subject": inputs["subject"], "body": inputs["body"]})
    try:
        parsed = json.loads(res.content.strip().strip("```json").strip("```"))
        return {"category": parsed.get("category", "")}
    except:
        return {"category": ""}

# Custom Evaluator function
def check_category_match(run, example) -> dict:
    # Get prediction and target reference
    prediction = run.outputs.get("category", "").lower()
    reference = example.outputs.get("category", "").lower()
    
    score = 1.0 if prediction == reference else 0.0
    return {
        "key": "category_accuracy",
        "score": score
    }

# Run evaluation harness!
print("Starting Email Classification Evaluation in LangSmith...")
results = evaluate(
    email_classifier_target,
    data="Email Classification Benchmark",
    evaluators=[check_category_match],
    experiment_prefix="email-classification-experiment"
)

### 2. Evaluating CodeWriterAgent code syntax correctness
Now let's write an evaluation function for our **CodeWriterAgent**. 

Here, our custom evaluator will check code compilation. It parses the generated code string output: if it compiles cleanly without throwing a `SyntaxError`, it returns a score of `1.0`. If compilation fails, it returns `0.0`.

In [ ]:
# Target Runner: Simulates generating code for prompt description
def code_generator_target(inputs: dict):
    llm = get_llm(model="gpt-4o-mini", temperature=0)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Write a python function for the user request. Output code only inside JSON object under key 'code'."),
        ("human", "Request: {task_description}")
    ])
    chain = prompt | llm
    res = chain.invoke({"task_description": inputs["task_description"]})
    try:
        parsed = json.loads(res.content.strip().strip("```json").strip("```"))
        return {"generated_code": parsed.get("code", "")}
    except:
        return {"generated_code": ""}

# Custom Compiler/Syntax Evaluator
def check_compilation_success(run, example) -> dict:
    code = run.outputs.get("generated_code", "")
    if not code.strip():
        return {"key": "syntax_correctness", "score": 0.0}
        
    try:
        compile(code, "<string>", "exec")
        score = 1.0
    except SyntaxError:
>       score = 0.0
        
    return {
        "key": "syntax_correctness",
        "score": score
    }

# Run evaluation harness!
print("Starting Code Generation Evaluation in LangSmith...")
results_code = evaluate(
    code_generator_target,
    data="Code Generator Benchmark",
    evaluators=[check_compilation_success],
    experiment_prefix="code-generation-syntax-check"
)

### Summary
Congratulations! You have completed the LangSmith testing and evaluation lab.

Open your **[LangSmith Dashboard](https://smith.langchain.com)**. Under **Projects**, check the tracing logs for your workflow nodes. Under **Datasets & Testing**, click on `Email Classification Benchmark` and `Code Generator Benchmark` to see the accuracy metrics, test cases, and comparative chart benchmarks across experiments!

In [ ]:
pip install langchain